# beta-VAE — Learning Basic Visual Concepts with a Constrained Variational Framework (2017)

_Papers · Generative Models_

**Put a single knob beta on the VAE's KL term: turn it above 1 and each latent dimension starts to capture one independent factor of the data.**

---

This notebook is a practice scaffold for the **beta-VAE — Learning Basic Visual Concepts with a Constrained Variational Framework (2017)** lesson. We build it step by step: the worked KL/loss example, a toy two-factor dataset, a plain VAE whose loss carries the beta knob, and a sweep of beta that trades reconstruction for disentanglement. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the beta-VAE in four steps: (1) the worked KL and beta-ELBO loss for a single latent dimension, (2) a toy dataset whose two independent factors are entangled in the pixels, (3) a plain VAE plus the scoring helpers, and (4) a sweep over beta that shows the reconstruction / disentanglement trade-off.

### Step 1 — The KL term and the beta-ELBO loss by hand

The beta-VAE loss is reconstruction cost plus `beta` times the KL of the latent posterior against the prior `N(0,1)`. The whole paper is that single `beta` multiplier. Here we take one latent dimension with `mu=1.0` and `log var=-0.5108`, compute its closed-form KL, and evaluate the loss at `beta=1` (an ordinary VAE) and `beta=4` to see how the knob raises the complexity penalty.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity-check the lesson's worked example.
mu0 = 1.0
logvar0 = -0.5108                          # sigma^2 = exp(-0.5108) = 0.6
sigma2_0 = math.exp(logvar0)

kl0 = -0.5 * (1 + logvar0 - mu0**2 - sigma2_0)   # closed-form KL for one dim
R0 = 10.0                                         # reconstruction cost (nats), given

print(f"worked example:  D_KL={kl0:.4f}")
print(f"  loss @ beta=1:  R + 1*D_KL = {R0 + 1 * kl0:.4f}")
print(f"  loss @ beta=4:  R + 4*D_KL = {R0 + 4 * kl0:.4f}")
# D_KL=0.5554 ; loss@1=10.5554 ; loss@4=12.2217

### Step 2 — A toy factored dataset with entangled pixels

Each image is a single Gaussian dot on a 12x12 grid. Its position is set by **two independent factors** — an angle and a radius — but those factors are mixed nonlinearly into the pixel values, so they are *entangled* in the raw data. The goal of disentanglement is to recover these two factors as separate latent dimensions. We return both the images and the true factor values so we can score that later.

In [ ]:
# Toy FACTORED dataset: 2 independent factors (angle, radius) ENTANGLED in pixels.
G = 12  # 12x12 image

def make_data(n=6000):
    a = torch.rand(n)                         # factor 1: angle in [0,1]
    b = torch.rand(n)                         # factor 2: radius in [0,1]
    th = a * math.pi
    r = 0.15 + 0.7 * b
    cx = 0.5 + 0.5 * r * torch.cos(th)        # dot center mixes BOTH factors nonlinearly
    cy = 0.5 + 0.5 * r * torch.sin(th)
    xs = torch.linspace(0, 1, G)
    gx, gy = xs.view(1, 1, G), xs.view(1, G, 1)
    img = torch.exp(-(((gx - cx.view(n, 1, 1))**2 + (gy - cy.view(n, 1, 1))**2)) / (2 * 0.01))
    return img.reshape(n, G * G), torch.stack([a, b], 1)   # images, true factors

X, FAC = make_data()
X, FAC = X.to(device), FAC.to(device)
print("images:", tuple(X.shape), " true factors:", tuple(FAC.shape))

### Step 3 — A plain VAE, the beta loss, and the scoring helpers

The `VAE` here is an ordinary encoder-decoder — beta does *not* live in the architecture. The `beta_elbo_loss` is where the paper's idea sits: it scales only the KL term by `beta`. `disentanglement_score` correlates each true factor with every latent mean and reports the average gap between the top and second correlation (large gap = one factor per latent), plus how many latent dimensions stay active. `train` runs Adam for a fixed number of epochs.

In [ ]:
# A plain VAE (identical to paper-vae). The beta lives only in the loss.
class VAE(nn.Module):
    def __init__(self, latent=4):
        super().__init__()
        self.e1 = nn.Linear(G * G, 256)
        self.mu = nn.Linear(256, latent)
        self.lv = nn.Linear(256, latent)
        self.d1 = nn.Linear(latent, 256)
        self.d2 = nn.Linear(256, G * G)

    def encode(self, x):
        h = F.relu(self.e1(x))
        return self.mu(h), self.lv(h)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(lv)        # reparameterize
        return torch.sigmoid(self.d2(F.relu(self.d1(z)))), mu, lv


def beta_elbo_loss(recon, x, mu, lv, beta):                        # <-- the whole paper: 'beta *'
    recon_term = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
    kl_term = (-0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1)).mean()
    return recon_term + beta * kl_term


def disentanglement_score(net):
    # For each true factor, correlate it with every latent mean dim. A disentangled code:
    # one strong correlation per factor, near-zero elsewhere. Score = avg gap (top - 2nd).
    net.eval()
    with torch.no_grad():
        _, mu, lv = net(X)
        mu = mu - mu.mean(0)
        L = mu.shape[1]
        corr = torch.zeros(2, L)
        for f in range(2):
            fac = FAC[:, f] - FAC[:, f].mean()
            for d in range(L):
                corr[f, d] = ((fac * mu[:, d]).mean() / (fac.std() * mu[:, d].std() + 1e-8)).abs()
        gaps = [(torch.sort(corr[f], descending=True)[0][:2].diff().abs().item()) for f in range(2)]
        active = int((torch.exp(0.5 * lv).mean(0) < 0.5).sum())     # dims the encoder actually uses
        rec, _, _ = net(X)
        recon = F.binary_cross_entropy(rec, X, reduction="sum").item() / len(X)
    return recon, sum(gaps) / 2, active


def train(beta, epochs=80, latent=4):
    torch.manual_seed(0)
    net = VAE(latent).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    net.train()
    for ep in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 256):
            xb = X[perm[i:i + 256]]
            recon, mu, lv = net(xb)
            loss = beta_elbo_loss(recon, xb, mu, lv, beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return net

### Step 4 — Sweep beta and watch the trade-off

Now we train the same VAE at `beta` = 1, 4, and 8 and read off reconstruction error, the disentanglement gap, and the number of active latents. `beta=1` is the plain VAE: best reconstruction but entangled. `beta=4` trades some reconstruction for genuine disentanglement — the paper's effect. `beta=8` over-presses the KL into posterior collapse, where latents go inactive and the gap crashes.

In [ ]:
# Sweep beta. beta=1 is the VAE; beta>1 is the new model.
print("Sweeping beta on the toy factored dataset:")
print("  beta   recon_BCE   disent_gap   active_latents")
for beta in (1.0, 4.0, 8.0):
    net = train(beta)
    recon, gap, active = disentanglement_score(net)
    tag = "VAE (baseline)" if beta == 1 else ("tuned beta>1" if beta == 4 else "too large -> collapse")
    print(f"  {beta:>4}   {recon:8.2f}   {gap:9.3f}   {active:^14d}  {tag}")

print("\nbeta=1: best reconstruction but ENTANGLED (low gap).")
print("beta=4: reconstruction worse, but DISENTANGLED (gap up) -- the paper's effect.")
print("beta=8: over-pressed -> posterior collapse (active->0), gap crashes.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_As we raise beta on the KL term, what happens to reconstruction and to disentanglement? Sweep beta in {1, 4, 8} on a toy 2-factor dataset._

### Step 1 — Rebuild the dataset and VAE, and define one beta run

This visualization cell is self-contained, so it re-imports torch and re-defines the same two-factor dataset and VAE from the reference implementation. The `run` function trains at a given `beta` and then computes both the reconstruction BCE and the disentanglement gap — the two numbers we want to compare across beta.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

G = 12
def make_data(n=6000):
    a, b = torch.rand(n), torch.rand(n)              # two independent factors: angle, radius
    th, r = a * math.pi, 0.15 + 0.7 * b
    cx = 0.5 + 0.5 * r * torch.cos(th)
    cy = 0.5 + 0.5 * r * torch.sin(th)
    xs = torch.linspace(0, 1, G)
    gx, gy = xs.view(1, 1, G), xs.view(1, G, 1)
    img = torch.exp(-(((gx - cx.view(n, 1, 1))**2 + (gy - cy.view(n, 1, 1))**2)) / (2 * 0.01))
    return img.reshape(n, G * G), torch.stack([a, b], 1)

X, FAC = make_data()


class VAE(nn.Module):
    def __init__(s, latent=4):
        super().__init__()
        s.e1 = nn.Linear(G * G, 256)
        s.mu = nn.Linear(256, latent)
        s.lv = nn.Linear(256, latent)
        s.d1 = nn.Linear(latent, 256)
        s.d2 = nn.Linear(256, G * G)

    def enc(s, x):
        h = F.relu(s.e1(x))
        return s.mu(h), s.lv(h)

    def forward(s, x):
        mu, lv = s.enc(x)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(lv)
        return torch.sigmoid(s.d2(F.relu(s.d1(z)))), mu, lv


def run(beta):
    torch.manual_seed(0)
    net = VAE()
    opt = torch.optim.Adam(net.parameters(), 2e-3)
    for ep in range(80):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 256):
            xb = X[perm[i:i + 256]]
            rec, mu, lv = net(xb)
            r = F.binary_cross_entropy(rec, xb, reduction="sum") / xb.size(0)
            kl = (-0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), 1)).mean()
            loss = r + beta * kl                              # beta * KL
            opt.zero_grad()
            loss.backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        rec, mu, lv = net(X)
        mu = mu - mu.mean(0)
        corr = torch.zeros(2, mu.shape[1])
        for f in range(2):
            fac = FAC[:, f] - FAC[:, f].mean()
            for d in range(mu.shape[1]):
                corr[f, d] = ((fac * mu[:, d]).mean() / (fac.std() * mu[:, d].std() + 1e-8)).abs()
        gap = sum(torch.sort(corr[f], descending=True)[0][:2].diff().abs().item() for f in range(2)) / 2
        recon = F.binary_cross_entropy(rec, X, reduction="sum").item() / len(X)
    return recon, gap

### Step 2 — Run the beta sweep and print the trade-off

With the machinery in place, we run beta = 1, 4, 8 and print the reconstruction BCE and disentanglement gap for each. Read it top to bottom: reconstruction best at beta=1 with an entangled code, disentanglement rising at beta=4, then posterior collapse crashing the gap at beta=8 — the appropriately-tuned beta>1 is the sweet spot.

In [ ]:
for beta in (1.0, 4.0, 8.0):
    recon, gap = run(beta)
    print(f"beta={beta:>4}: recon_BCE={recon:6.2f}  disent_gap={gap:.3f}")
# beta=1: recon best, code entangled (low gap). beta=4: recon worse, DISENTANGLED (gap up).
# beta=8: posterior collapse -> gap crashes. 'appropriately tuned' beta>1 is the sweet spot.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You train the same VAE on the toy factored data with $\beta=1$ and again
            with $\beta=4$, then compare reconstruction error and a disentanglement score. What do you
            expect for each, and what does the comparison prove $\beta$ is doing?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Hold architecture, optimizer, data, seed, and latent size fixed; change only $\beta$ from $1$ to $4$. — _An honest ablation changes exactly one thing &mdash; the KL weight &mdash; so any difference is attributable to $\beta$._
- Expect reconstruction error to rise at $\beta=4$ (codes carry less information, rebuilds soften). — _A larger $\beta$ taxes information in the code, so the decoder gets a less detailed $z$ and reconstructs less sharply._
- Expect the disentanglement score to rise at $\beta=4$ (each factor concentrates on one latent). — _The tighter bottleneck makes matching the factorised prior $N(0,I)$ cheaper than smearing factors across dimensions, so factors align to axes._
- Conclude $\beta$ trades reconstruction for disentanglement &mdash; the central claim. — _The two curves moving in opposite directions is the paper's result, isolated to the one weight._

**Answer:** Reconstruction gets worse at $\beta=4$ while disentanglement gets better:
                 the latents go from mixing both factors (entangled) to each capturing one factor. This
                 isolates $\beta$ as the disentanglement knob &mdash; it buys factor-aligned latents by
                 spending reconstruction fidelity. In our run, reconstruction BCE rose from $\approx 13.7$
                 ($\beta{=}1$) to $\approx 18.2$ ($\beta{=}4$) while the disentanglement gap rose from
                 $\approx 0.17$ to $\approx 0.43$. (Our tiny run, not the paper's numbers.)

</details>

**Problem 2.** For one latent dimension the encoder outputs $\mu = 1.0$ and $\log\sigma^2 = -0.5108$
            (so $\sigma^2 = 0.6$), and the reconstruction cost on this input is $R = 10.0$ nats. Compute the
            KL for this dimension and the negative-$\beta$-ELBO loss at $\beta=1$ and $\beta=4$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- KL (closed form): $D_{KL} = -\tfrac{1}{2}(1 + \log\sigma^2 - \mu^2 - \sigma^2) = -\tfrac{1}{2}(1 - 0.5108 - 1.0 - 0.6)$. — _The closed-form Gaussian KL against $N(0,1)$; plug $\mu^2=1.0$, $\sigma^2=0.6$, $\log\sigma^2=-0.5108$._
- Evaluate: $-\tfrac{1}{2}(-1.1108) = 0.5554$, so $D_{KL} = 0.5554$. — _Positive because this code ($\mu=1$, $\sigma^2=0.6$) is off-center and narrower than the prior._
- Loss $= R + \beta D_{KL}$. At $\beta=1$: $10.0 + 0.5554 = 10.5554$. At $\beta=4$: $10.0 + 4(0.5554) = 12.2217$. — _The $\beta$-weighted objective: $\beta$ scales only the KL penalty, leaving reconstruction untouched._

**Answer:** $D_{KL} \approx 0.5554$. Loss $= 10.5554$ at $\beta=1$ and $12.2217$ at $\beta=4$.
                 Raising $\beta$ added $3\times D_{KL} \approx 1.67$ nats of penalty for using this code,
                 with no change to reconstruction &mdash; so training is pushed harder to shrink the KL
                 ($\mu\to 0$, $\sigma\to 1$) or abandon the dimension. These are the worked-example
                 numbers recomputed in the notebook.

</details>

**Problem 3.** A teammate reports: "I cranked $\beta$ to $50$ and my disentanglement score went to almost zero
            and reconstructions are blank. The paper must be wrong." What actually happened, and what is the
            fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize posterior collapse: at very large $\beta$ the KL term dominates so heavily that the cheapest solution is $q_\phi(z\mid x)\approx p(z)$ for every input. — _When the information tax is enormous, the encoder pays least by carrying no information &mdash; $\mu\to 0$, $\sigma\to 1$ everywhere &mdash; so the code is constant across inputs._
- Note that a constant code can encode no factor, so disentanglement (one factor per latent) is undefined/zero, and the decoder, seeing no signal, outputs the data mean (blank-ish). — _Disentanglement requires the latents to vary with the factors; a collapsed code does not vary, so there is nothing to align._
- Fix: $\beta$ is "appropriately tuned," not "as large as possible." Sweep $\beta$ and pick the sweet spot; optionally anneal $\beta$ up gradually (KL warmup) so the encoder learns to use the code before the tax bites. — _The paper's claim is about a tuned $\beta\gt 1$; the trade-off is non-monotone, so the operating point matters._

**Answer:** $\beta=50$ caused posterior collapse: the KL penalty so dominates that the encoder
                 outputs the prior for everything, the code carries no information, disentanglement is
                 ~$0$, and reconstructions go blank. The paper is not wrong &mdash; it says
                 appropriately tuned $\beta\gt 1$. The fix is to sweep $\beta$ for the sweet spot
                 (and optionally anneal it up). In our run, $\beta=8$ already shows the onset: latents
                 collapse (active dims $\to 0$) and the disentanglement gap crashes to $\approx 0.02$.

</details>